In [85]:
import pandas as pd

url = "https://raw.githubusercontent.com/plotly/datasets/master/diabetes.csv"
data = pd.read_csv(url)

data.head(2)

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0


In [86]:
X = data.drop(columns=["Outcome"])
Y = data["Outcome"]

display(X.head(2))
display(Y.head(2))


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age
0,6,148,72,35,0,33.6,0.627,50
1,1,85,66,29,0,26.6,0.351,31


,Outcome
0,1
1,0


In [87]:
from sklearn.model_selection import train_test_split

X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42, stratify=Y)

print("Training patients:", len(X_train))
print("Testing patients:", len(X_test))


Training patients: 614
Testing patients: 154


In [88]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)



In [89]:
from sklearn.svm import SVC

model = SVC(kernel="linear")
model.fit(X_train_scaled, Y_train)

SVC(kernel='linear')

In [90]:
Y_pred = model.predict(X_test_scaled)

results = pd.DataFrame({ "Predicted": Y_pred, "Actual": Y_test})

display(results.head(10))

correct = (results["Predicted"] == results["Actual"]).sum()
print(f"Correct predictions: {correct} out of {len(results)}")

,Predicted,Actual
44,1,0
672,0,0
700,0,0
630,0,1
81,0,0
389,0,0
387,0,1
408,1,1
163,0,0
335,1,0


Correct predictions: 111 out of 154


In [91]:
from sklearn.metrics import accuracy_score, recall_score

accuracy = accuracy_score(Y_test, Y_pred)
recall = recall_score(Y_test, Y_pred)

print(f"Overall accuracy: {accuracy:.1%}")
print(f"Recall for positive cases: {recall:1%}")


Overall accuracy: 72.1%
Recall for positive cases: 51.851852%


In [92]:
from sklearn.metrics import confusion_matrix

scores = model.decision_function(X_test_scaled)

for cutoff in [0, -0.796093]:
  predictions  = (scores > cutoff).astype(int)
  tn, fp, fn, tp = confusion_matrix(Y_test, predictions, labels=[0, 1]).ravel()

  print(f"Cutoff {cutoff}")
  print(f" Positive cases caught: {tp} out of {tp + fn}")
  print(f" Positive cases missed: {fn}")
  print(f" False alarms: {fp}")
  print(f" Recall: {tp / (tp + fn):.1%}")

Cutoff 0
 Positive cases caught: 28 out of 54
 Positive cases missed: 26
 False alarms: 17
 Recall: 51.9%
Cutoff -0.796093
 Positive cases caught: 47 out of 54
 Positive cases missed: 7
 False alarms: 30
 Recall: 87.0%


In [93]:
cutoff = -0.05

results = pd.DataFrame({ "Score": scores, "Predicted": (scores > cutoff).astype(int), "Actual": Y_test})

false_positives = results[(results["Predicted"] ==1) & (results["Actual"] == 0)]

display(false_positives.sort_values("Score").head(5))

display(false_positives[false_positives["Score"] >= 0.3])

,Score,Predicted,Actual
437,-0.008405,1,0
517,-0.006018,1,0
486,0.028635,1,0
345,0.053271,1,0
244,0.074021,1,0


,Score,Predicted,Actual
44,0.468505,1,0
335,1.102437,1,0
660,0.646205,1,0
260,1.182707,1,0
153,0.318990,1,0
469,1.401259,1,0
558,0.308088,1,0
436,0.769870,1,0
489,1.647431,1,0
247,1.158969,1,0


In [94]:
patient = X_test.iloc[[0]]
display(patient)

patient_scaled = scaler.transform(patient)
score = model.decision_function(patient_scaled)[0]
print("Model score:", score)

print("Model prediction:", int(score > -0.796093))
print("Recorded outcome:", Y_test.iloc[0])

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age
44,7,159,64,0,0,27.4,0.294,40


Model score: 0.4685052866559356
Model prediction: 1
Recorded outcome: 0


In [95]:
def predict_patient(patient):
  patient_scaled = scaler.transform(patient)
  score = model.decision_function(patient_scaled)[0]
  prediction = int(score > -0.796093)
  return prediction

In [96]:
patient = X_test.iloc[[1]]

print("Predicted:", predict_patient(patient))
print("Actual:", Y_test.iloc[1])

Predicted: 0
Actual: 0


In [98]:
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.pipeline import make_pipeline

practice_model = make_pipeline(StandardScaler(), SVC(kernel="linear"))
folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

validation_scores = cross_val_predict(practice_model, X_train, Y_train, cv=folds, method="decision_function")

print("Patients scored:", len(validation_scores))

Patients scored: 614


In [100]:
print(ranked.iloc[0].to_string())


Cutoff               -0.796093
Recall                0.803738
Positives caught    172.000000
False alarms        130.000000


In [101]:
import joblib
from google.colab import files

saved_predictor = {
    "model": model,
    "scaler": scaler,
    "cutoff": -0.8,
    "features": X_train.columns.tolist(),
    "comparison": ">="
}

joblib.dump(saved_predictor, "diabetes_predictor.joblib")
files.download("diabetes_predictor.joblib")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [97]:
example_patient = pd.DataFrame([{
    "Pregnancies": 6,
    "Glucose": 130,
    "BloodPressure": 70,
    "SkinThickness": 12,
    "Insulin": 85,
    "BMI": 40.0,
    "DiabetesPedigreeFunction": 0.40,
    "Age": 19
}])

print("Predicted outcome:", predict_patient(example_patient))

Predicted outcome: 1
